# MQAR: State Size vs. Recall Accuracy (Mamba2 / Gated DeltaNet 2)

Single-file, Colab-importable recreation of the central **Zoology** experiment
(Arora, Eyuboglu, et al., *"Zoology: Measuring and Improving Recall in Efficient
Language Models"*, https://github.com/HazyResearch/zoology) for **bounded-state**
sub-quadratic sequence mixers.

This is `exp003`: it extends the Mamba2 notebook (`mqar_exp002_mamba2.ipynb`) with a
**selectable sequence mixer**. Set `MIXER` in the Config cell:

* `"mamba2"` — the pure-PyTorch Mamba2 SSM from the previous notebook (no CUDA kernels).
* `"gdn2"` — **Gated DeltaNet 2**, imported directly from
  [flash-linear-attention](https://github.com/fla-org/flash-linear-attention)
  (`fla.layers.GatedDeltaNet2`). Requires a CUDA GPU + Triton.

The MQAR task, training loop, logging harness, and the **state-size x-axis** are shared;
only the sequence mixer (and its state-size formula) changes.

### What it does
* Trains one decoder-only model per width in `D_MODELS` (mixer-specific defaults).
* Each width maps to a point on the **x-axis (state size)** — the bounded recurrent
  state, in bytes, **independent of sequence length**:
  * Mamba2: SSM state `expand·d_model·d_state` per layer.
  * Gated DeltaNet 2: per-head key×value state `num_heads·head_dim·head_v_dim` per layer.
* Every run is **fully reproducible** (global determinism + per-run seeds) and the
  **complete nested config is saved to Weights & Biases**, with library/GPU versions and
  a dataset fingerprint.
* A final summary run logs the recreated curve as a matplotlib figure, a `wandb.Table`,
  and a native W&B line plot.

### How to run in Colab
1. Set the runtime to **GPU** (Runtime → Change runtime type → GPU). `gdn2` *requires* a
   GPU; `mamba2` also runs on CPU, just slower.
2. Set `MIXER`, and your `WANDB_ENTITY` / `WANDB_PROJECT`, in the **Config** cell
   (or set `WANDB_MODE="disabled"` to run without W&B).
3. Run all cells top to bottom. The `gdn2` mixer auto-installs
   `flash-linear-attention` on first use.

## 0. Dependencies
Torch / numpy / matplotlib / einops / tqdm ship with Colab; we ensure `wandb` (and
`einops`). When `MIXER="gdn2"`, the Config cell additionally installs
**`flash-linear-attention`** (`fla`), which provides `fla.layers.GatedDeltaNet2` and pulls
in Triton — this needs a CUDA GPU. The `mamba2` mixer needs no CUDA SSM kernels
(`mamba_ssm`, `causal_conv1d`, `triton`); it is pure PyTorch.

In [ ]:
import importlib
import subprocess
import sys


def _ensure(pip_name: str, import_name: str = None):
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)


for _pkg, _imp in [("wandb", "wandb"), ("einops", "einops"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]:
    _ensure(_pkg, _imp)

## 1. Configuration
All experiment knobs live here, starting with **`MIXER`** (`"mamba2"` or `"gdn2"`), which
selects the sequence mixer and its width sweep (`D_MODELS`). The MQAR difficulty and the
training hyper-parameters are exposed as plain constants so they are easy to edit in Colab.

In [ ]:
import os
import json
import math
import random
import platform
import hashlib
import uuid
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from tqdm.auto import tqdm

# ---- Sequence mixer ---------------------------------------------------------------
# Pick the sub-quadratic sequence mixer for this run. Both share the same MQAR task,
# training/logging harness and state-size x-axis.
#   "mamba2" -> pure-PyTorch Mamba2 (no CUDA kernels; runs on CPU or GPU)
#   "gdn2"   -> Gated DeltaNet 2 from flash-linear-attention (fla.layers.GatedDeltaNet2;
#               needs a CUDA GPU + Triton for the chunk kernel)
MIXER = "gdn2"               # "mamba2" | "gdn2"

# gdn2 needs flash-linear-attention; install it now (uses _ensure from the Dependencies cell).
if MIXER == "gdn2":
    _ensure("flash-linear-attention", "fla")   # provides fla.layers.GatedDeltaNet2 (pulls triton)

# ---- Weights & Biases -------------------------------------------------------------
WANDB_PROJECT = "zoology-mqar"
WANDB_ENTITY = "rkstgr"          # <-- set to your W&B entity (username/team), or leave None
WANDB_MODE = "online"        # "online" | "offline" | "disabled"

# ---- The sweep: explicit model widths ---------------------------------------------
# One run per d_model. Each width maps to a state size (the x-axis). The mapping is
# mixer-specific:
#   mamba2: state = 4 * n_layers * (expand * d_model) * d_state
#   gdn2:   state = 4 * n_layers * num_heads * head_dim * head_v_dim, with
#           num_heads = d_model // gdn2_head_dim (so the per-head K x V state matrices
#           grow with width). gdn2 widths are multiples of gdn2_head_dim (=64) so the
#           Triton kernel gets a kernel-friendly head dim and an integral head count.
# Neither state depends on sequence length (bounded recurrent state) -- width is the knob.
D_MODELS = {
    "mamba2": [8, 16, 64, 128, 192],
    "gdn2":   [64, 128, 192, 256, 320],
}[MIXER]
N_POINTS = len(D_MODELS)

# ---- Per-run learning rate: clamped 1/width scaling -------------------------------
# Wider models (larger state) get a proportionally lower peak LR. Anchored so the
# mid-scale run keeps the LR that was known to work well.
LR_REF = 1e-3                # reference peak LR ...
D_REF = 49                   # ... at this reference model width (the mid-scale run)
LR_MIN = 5e-4                # clamp floor: keep the widest models from going too slow
LR_MAX = 1.5e-3              # clamp ceiling: keep the narrowest models from running hot

# ---- Global reproducibility seed --------------------------------------------------
SEED = 123


@dataclass
class MQARTaskConfig:
    """Fixed MQAR difficulty (mirrors a canonical zoology train/test segment)."""
    vocab_size: int = 8192
    input_seq_len: int = 128
    num_kv_pairs: int = 8
    power_a: float = 0.01
    random_non_queries: bool = True
    num_train_examples: int = 50_000
    num_test_examples: int = 1_000


@dataclass
class ModelConfig:
    """Decoder-only sequence model with a selectable sub-quadratic sequence mixer."""
    d_model: int = 128
    n_layers: int = 2
    mixer: str = MIXER          # "mamba2" | "gdn2"
    # ---- Mamba2 sequence-mixer hyper-parameters ----
    d_state: int = 128          # SSM state dimension N (the recurrent-memory knob)
    d_conv: int = 4             # depthwise causal conv width
    expand: int = 2             # inner expansion: d_inner = expand * d_model
    headdim: int = 64           # target head dim (auto-shrunk to divide d_inner)
    ngroups: int = 1            # B/C groups (shared across heads when ngroups=1)
    # ---- Gated DeltaNet 2 sequence-mixer hyper-parameters (fla.layers.GatedDeltaNet2) ----
    gdn2_head_dim: int = 64           # per-head K dim (also V dim when expand_v=1); the memory knob
    gdn2_num_heads: int = None        # default: d_model // gdn2_head_dim
    gdn2_expand_v: float = 1.0        # value expansion: head_v_dim = head_dim * expand_v
    gdn2_mode: str = "chunk"          # "chunk" (training) | "fused_recurrent"
    gdn2_use_short_conv: bool = True  # short causal conv on q/k/v (as in fla)
    gdn2_conv_size: int = 4
    gdn2_allow_neg_eigval: bool = False
    # ---- shared ----
    mlp_hidden_mult: int = 4
    vocab_size: int = 8192
    max_position_embeddings: int = 128
    resid_dropout: float = 0.0
    embed_dropout: float = 0.0
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    learnable_word_embeddings: bool = True
    block_type: str = "TransformerBlock"
    sequence_mixer: str = ""    # set in __post_init__ from `mixer`
    state_mixer: str = "zoology.mixers.mlp.MLP"

    def __post_init__(self):
        self.sequence_mixer = {
            "mamba2": "zoology.mixers.mamba2.Mamba2",
            "gdn2": "fla.layers.gdn2.GatedDeltaNet2",
        }[self.mixer]
        if self.gdn2_num_heads is None:
            self.gdn2_num_heads = max(1, self.d_model // self.gdn2_head_dim)


@dataclass
class TrainParams:
    """Training hyper-parameters (zoology/train.py + warmup, grad-clip, patience).

    The peak learning rate is *not* fixed here -- it is set per run by `lr_for_d_model`
    (clamped 1/width). The schedule is a linear warmup over `warmup_epochs` followed by
    per-step cosine decay to 0.
    """
    max_epochs: int = 32
    weight_decay: float = 0.1
    batch_size: int = 256
    test_batch_size: int = 256
    warmup_epochs: float = 1.0           # linear LR warmup over this many epochs
    grad_clip: float = 1.0               # max gradient norm (<= 0 disables clipping)
    early_stopping_metric: str = "valid/accuracy"
    early_stopping_threshold: float = 0.99   # stop early once the task is solved
    patience: int = 5                   # stop if valid/accuracy hasn't improved for N epochs
    seed: int = SEED


def lr_for_d_model(d_model: int) -> float:
    """Clamped 1/width peak LR: lr = LR_REF * (D_REF / d_model), clamped to [LR_MIN, LR_MAX]."""
    return float(min(LR_MAX, max(LR_MIN, LR_REF * (D_REF / d_model))))


TASK = MQARTaskConfig()
TRAIN = TrainParams()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}  (mixer={MIXER!r})")
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- runs will be slow. In Colab: Runtime > Change runtime type > GPU.")
if MIXER == "gdn2" and DEVICE == "cpu":
    print("WARNING: MIXER='gdn2' uses flash-linear-attention's Triton kernels, which require a CUDA GPU. "
          "On CPU the run will fail -- switch the Colab runtime to GPU, or set MIXER='mamba2'.")


def set_determinism(seed: int):
    """Make a run reproducible end-to-end (mirrors zoology.utils.set_determinism + CUDA)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

## 2. MQAR data generation
Copied (near-verbatim) from `zoology/data/multiquery_ar.py`. Each example places
`num_kv_pairs` key→value bindings up front, then queries a subset of those keys; the
model must recall the matching value. Non-query positions are filled with random
distractor tokens. Labels are `-100` (ignored by the loss/metric) except at answer
positions. The large vocabulary (8192) is, per the paper, important for separating
architectures.

In [ ]:
def multiquery_ar(
    vocab_size: int,
    num_examples: int,
    input_seq_len: int,
    seed: int,
    power_a: float = 0.01,
    num_kv_pairs: int = 8,
    num_passes: int = 1,
    random_non_queries: bool = True,
):
    """Generate (inputs, labels) for the multi-query associative recall task.

    Verbatim port of zoology.data.multiquery_ar.multiquery_ar (HazyResearch/zoology).
    """
    assert input_seq_len % 2 == 0, "input_seq_len must be even"
    assert vocab_size > input_seq_len
    assert num_kv_pairs * 2 * num_passes + num_kv_pairs * 2 <= input_seq_len

    np.random.seed(seed)

    context_size = num_kv_pairs * 2 * num_passes

    # keys / values are drawn from disjoint halves of the vocabulary
    key_vocab_size = vocab_size // 2
    key_choices = np.arange(1, key_vocab_size)
    value_choices = np.arange(key_vocab_size, vocab_size)

    keys_unshuffled = np.tile(key_choices, (num_examples, 1))
    keys = np.apply_along_axis(np.random.choice, 1, keys_unshuffled, replace=False, size=num_kv_pairs)

    values_unshuffled = np.tile(value_choices, (num_examples, 1))
    values = np.apply_along_axis(np.random.choice, 1, values_unshuffled, replace=False, size=num_kv_pairs)

    kvs = np.zeros((num_examples, context_size), dtype=np.int64)
    kvs[:, 0::2] = keys
    kvs[:, 1::2] = values
    kvs = np.tile(kvs, (1, num_passes))

    # power-law gap distribution between a key-value pair and its query
    space = (input_seq_len - context_size) // 2
    p = power_a * np.arange(1, space + 1) ** (power_a - 1)
    p = p / p.sum()

    x = np.stack([np.arange(space, dtype=int)] * num_examples)
    gaps = np.apply_along_axis(np.random.choice, axis=1, arr=x, replace=False, p=p, size=num_kv_pairs)

    queries = np.zeros((num_examples, input_seq_len - context_size + 1), dtype=np.int64)
    np.put_along_axis(queries, (gaps * 2), values=keys, axis=1)
    examples = np.concatenate([kvs, queries], axis=1)

    labels = np.full((num_examples, input_seq_len + 1), -100, dtype=np.int64)
    np.put_along_axis(labels, (gaps * 2) + context_size + 1, values=values, axis=1)

    inputs, labels = torch.tensor(examples[:, :-1]), torch.tensor(labels[:, 1:])

    if random_non_queries:
        inputs[inputs == 0] = torch.randint(vocab_size, size=inputs.shape)[inputs == 0]

    return inputs, labels


def build_dataloaders(task: MQARTaskConfig, seed: int):
    """Build reproducible train/test loaders with disjoint train/test seeds."""
    MAX_SEED = 2 ** 32
    rng = np.random.RandomState(seed)
    train_seed = int(rng.randint(0, MAX_SEED // 2))
    test_seed = int(rng.randint(MAX_SEED // 2, MAX_SEED))

    train_inputs, train_labels = multiquery_ar(
        vocab_size=task.vocab_size, num_examples=task.num_train_examples,
        input_seq_len=task.input_seq_len, seed=train_seed,
        power_a=task.power_a, num_kv_pairs=task.num_kv_pairs,
        random_non_queries=task.random_non_queries,
    )
    test_inputs, test_labels = multiquery_ar(
        vocab_size=task.vocab_size, num_examples=task.num_test_examples,
        input_seq_len=task.input_seq_len, seed=test_seed,
        power_a=task.power_a, num_kv_pairs=task.num_kv_pairs,
        random_non_queries=task.random_non_queries,
    )

    # fingerprint the test set so dataset identity is verifiable across machines
    fingerprint = hashlib.md5(test_inputs.numpy().tobytes() + test_labels.numpy().tobytes()).hexdigest()

    train_ds = torch.utils.data.TensorDataset(train_inputs, train_labels)
    test_ds = torch.utils.data.TensorDataset(test_inputs, test_labels)

    g = torch.Generator().manual_seed(seed)  # reproducible shuffling
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=TRAIN.batch_size, shuffle=True, generator=g)
    test_dl = torch.utils.data.DataLoader(test_ds, batch_size=TRAIN.test_batch_size, shuffle=False)
    return train_dl, test_dl, fingerprint

## 3. Model — decoder-only, selectable mixer
A faithful re-implementation of zoology's `LanguageModel`:
token + learnable absolute position embeddings → `n_layers` pre-norm blocks of
(sequence-mixer + `MLP` state-mixer) → final LayerNorm → weight-tied LM head.
Initialization (`std=0.02`, residual projections scaled by `1/sqrt(2·n_layers)`) matches
`zoology/model.py`. The sequence mixer is chosen by `cfg.mixer`.

**Mamba2 mixer** (`MIXER="mamba2"`). Pure-PyTorch SSD scan + causal depthwise conv,
mirroring `zoology/mixers/mamba2.py`: `in_proj → [z, xBC, dt]`, `conv → SiLU → split x,B,C`,
the SSD recurrence in its exact O(L²) form, gated-RMSNorm with `z`, then `out_proj`. State
per layer is `expand·d_model·d_state` floats.

**Gated DeltaNet 2 mixer** (`MIXER="gdn2"`). A thin wrapper (`GatedDeltaNet2Mixer`) around
`fla.layers.GatedDeltaNet2` (flash-linear-attention) exposing the same `(b,l,d)→(b,l,d)`
interface plus a `state_size()` method, so it drops straight into the same Block/harness.
We set `num_heads = d_model // head_dim` (head_dim=64), `expand_v=1.0`, short conv on,
`mode="chunk"`. fla initializes its own parameters (we skip the generic init for its
sub-modules). The recurrent state is a per-head key×value matrix; summed over heads it is
`num_heads·head_dim·head_v_dim` floats per layer.

**State size** (the x-axis) sums the per-layer recurrent state over layers, ×4 bytes
(float32). It does **not** depend on sequence length — the bounded-state property that
distinguishes these mixers from attention. (Small conv states are omitted, as in zoology.)

In [ ]:
class RMSNormGated(nn.Module):
    """Gated RMSNorm used by Mamba2 (mirrors mamba_ssm RMSNormGated, norm_before_gate=False).

    When a gate `z` is given, normalizes `x * silu(z)` (gate-before-norm). We use a single
    group (ngroups=1), so the norm is taken over the full d_ssm dimension.
    """
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x, z=None):
        if z is not None:
            x = x * F.silu(z)
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * self.weight


def ssd_forward(x, dt, A, B, C, D):
    """Exact (single-chunk, quadratic) Mamba2 state-space-dual scan, in pure PyTorch.

    Evaluates the SSD recurrence
        h_t = exp(dt_t * A) * h_{t-1} + dt_t * B_t * x_t ,    y_t = C_t . h_t + D * x_t
    in closed form as
        y_t = sum_{s<=t} exp(cumsum_t - cumsum_s) * dt_s * (C_t . B_s) * x_s  +  D * x_t ,
    where cumsum is the cumulative sum of dt*A along the sequence. For our seq_len (128)
    the O(L^2) form is exact, simple and fast, and needs no CUDA kernels (mamba_ssm /
    causal_conv1d). It is mathematically identical to the chunked `mamba_chunk_scan_combined`
    used in the official implementation.

    Shapes:  x (b,l,h,p)  dt (b,l,h)  A (h,)  B,C (b,l,g,n)  D (h,)  ->  y (b,l,h,p)
    """
    b, l, h, p = x.shape
    g = B.shape[2]
    if g != h:                                   # broadcast B/C groups across heads (ngroups=1)
        B = B.repeat_interleave(h // g, dim=2)
        C = C.repeat_interleave(h // g, dim=2)

    dA = dt * A                                  # (b,l,h)  per-step log-decay (<= 0)
    dA_cumsum = torch.cumsum(dA, dim=1)          # (b,l,h)
    cs = rearrange(dA_cumsum, "b l h -> b h l")
    # decay[t,s] = exp(cumsum_t - cumsum_s) for s <= t, else 0 (masked to -inf before exp)
    decay = cs.unsqueeze(-1) - cs.unsqueeze(-2)  # (b,h,t,s)
    causal = torch.tril(torch.ones(l, l, dtype=torch.bool, device=x.device))
    decay = decay.masked_fill(~causal, float("-inf")).exp()

    CB = torch.einsum("blhn,bshn->bhls", C, B)   # (b,h,t,s)  C_t . B_s
    M = CB * decay
    xdt = x * dt.unsqueeze(-1)                    # fold dt into the input
    y = torch.einsum("bhls,bshp->blhp", M, xdt)  # (b,l,h,p)
    y = y + x * D.view(1, 1, h, 1)               # D skip connection
    return y


def _valid_headdim(d_inner: int, target: int = 64) -> int:
    """Largest head dim <= target that divides d_inner (keeps nheads integral; state size is fixed)."""
    hd = min(target, d_inner)
    while d_inner % hd != 0:
        hd -= 1
    return hd


class Mamba2(nn.Module):
    """Mamba2 sequence mixer (mirrors zoology.mixers.mamba2.Mamba2 / mamba_ssm Mamba2).

    Pure-PyTorch: the SSD scan and the causal depthwise conv are implemented without the
    CUDA kernels so this runs anywhere. Dimensions follow the reference exactly:
        d_inner  = expand * d_model
        nheads   = d_inner // headdim
        conv_dim = d_inner + 2 * ngroups * d_state
        in_proj -> [ z (d_inner), xBC (conv_dim), dt (nheads) ]
    """
    def __init__(self, d_model: int, d_state: int = 128, d_conv: int = 4, expand: int = 2,
                 headdim: int = 64, ngroups: int = 1, dt_min: float = 1e-3, dt_max: float = 0.1,
                 dt_init_floor: float = 1e-4, A_init_range=(1.0, 16.0), conv_bias: bool = True,
                 bias: bool = False, layer_idx: int = None):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = expand * d_model
        self.d_ssm = self.d_inner                       # no separate gated-MLP branch (d_ssm == d_inner)
        self.ngroups = ngroups
        self.headdim = _valid_headdim(self.d_inner, headdim)
        assert self.d_inner % self.headdim == 0, "d_inner must be divisible by headdim"
        self.nheads = self.d_inner // self.headdim
        self.layer_idx = layer_idx

        self.conv_dim = self.d_ssm + 2 * self.ngroups * self.d_state
        d_in_proj = 2 * self.d_inner + 2 * self.ngroups * self.d_state + self.nheads
        self.in_proj = nn.Linear(d_model, d_in_proj, bias=bias)

        self.conv1d = nn.Conv1d(
            in_channels=self.conv_dim, out_channels=self.conv_dim, bias=conv_bias,
            kernel_size=d_conv, groups=self.conv_dim, padding=d_conv - 1,
        )
        self.act = nn.SiLU()

        # dt bias: inverse-softplus of dt ~ exp(U[log dt_min, log dt_max]) (mamba_ssm init)
        dt = torch.exp(torch.rand(self.nheads) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        dt = dt.clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        self.dt_bias = nn.Parameter(inv_dt)

        # A (one scalar per head), stored in log space; A = -exp(A_log) < 0 -> stable decay
        A = torch.empty(self.nheads).uniform_(*A_init_range)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.nheads))

        self.norm = RMSNormGated(self.d_ssm, eps=1e-5)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=bias)

    def forward(self, u):
        b, l, _ = u.shape
        zxbcdt = self.in_proj(u)
        z, xBC, dt = torch.split(zxbcdt, [self.d_inner, self.conv_dim, self.nheads], dim=-1)

        # causal depthwise conv over the sequence (trim the right padding), then SiLU
        xBC = self.act(self.conv1d(xBC.transpose(1, 2))[..., :l].transpose(1, 2))
        x, B, C = torch.split(
            xBC, [self.d_ssm, self.ngroups * self.d_state, self.ngroups * self.d_state], dim=-1)

        x = rearrange(x, "b l (h p) -> b l h p", p=self.headdim)
        B = rearrange(B, "b l (g n) -> b l g n", g=self.ngroups)
        C = rearrange(C, "b l (g n) -> b l g n", g=self.ngroups)
        dt = F.softplus(dt + self.dt_bias)              # (b,l,nheads), positive timestep
        A = -torch.exp(self.A_log)                      # (nheads,)

        y = ssd_forward(x, dt, A, B, C, self.D)         # (b,l,h,p)
        y = rearrange(y, "b l h p -> b l (h p)")
        y = self.norm(y, z)                             # gated RMSNorm
        return self.out_proj(y)

    def state_size(self, sequence_length: int = 2048) -> int:
        # Recurrent SSM state h has shape (nheads, headdim, d_state) -> d_inner * d_state floats.
        # Independent of sequence length (bounded state). Equals zoology's 2 * d_model * d_state
        # for expand=2. (The small conv state, conv_dim * (d_conv-1), is omitted, as in zoology.)
        return self.d_inner * self.d_state


class GatedDeltaNet2Mixer(nn.Module):
    """Gated DeltaNet 2 sequence mixer -- thin wrapper over fla.layers.GatedDeltaNet2.

    Imports and uses flash-linear-attention's layer directly (per the fla README), and
    exposes the same (b,l,d) -> (b,l,d) interface and `state_size()` method as the local
    Mamba2 mixer, so it drops into the same Block / LanguageModel harness.

    Dimensions: q/k use `num_heads` heads of `head_dim`; v uses `head_v_dim = head_dim *
    expand_v`. The recurrent state is a per-head key x value matrix of shape
    (num_heads, head_dim, head_v_dim); summed over heads it has
    `num_heads * head_dim * head_v_dim` elements per layer, independent of sequence length.
    The fla layer runs its Triton chunk/fused_recurrent kernels (CUDA only).
    """
    def __init__(self, d_model: int, head_dim: int = 64, num_heads: int = None,
                 expand_v: float = 1.0, mode: str = "chunk", use_short_conv: bool = True,
                 conv_size: int = 4, allow_neg_eigval: bool = False, layer_idx: int = None):
        super().__init__()
        from fla.layers import GatedDeltaNet2   # lazy import: only needed for the gdn2 mixer
        if num_heads is None:
            num_heads = max(1, d_model // head_dim)
        self.d_model = d_model
        self.head_dim = head_dim
        self.head_v_dim = int(head_dim * expand_v)
        self.num_heads = num_heads
        self.num_v_heads = num_heads                    # we keep num_v_heads == num_heads
        self.gdn2 = GatedDeltaNet2(
            hidden_size=d_model, head_dim=head_dim, num_heads=num_heads, expand_v=expand_v,
            mode=mode, use_short_conv=use_short_conv, conv_size=conv_size,
            allow_neg_eigval=allow_neg_eigval, layer_idx=layer_idx,
        )

    def forward(self, u):
        # fla layers return (output, attentions, past_key_values); we only need the output.
        out, *_ = self.gdn2(u)
        return out

    def state_size(self, sequence_length: int = 2048) -> int:
        # Per-head delta-rule state S has shape (head_dim, head_v_dim) for each of the
        # num_v_heads value heads -> num_v_heads * head_dim * head_v_dim floats per layer.
        # Independent of sequence length (bounded state).
        return self.num_v_heads * self.head_dim * self.head_v_dim


class MLP(nn.Module):
    """Feed-forward state-mixer (zoology.mixers.mlp.MLP)."""
    def __init__(self, d_model: int, hidden_mult: int = 4, activation=F.gelu):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model * hidden_mult)
        self.activation = activation
        self.fc2 = nn.Linear(d_model * hidden_mult, d_model)

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))


def _make_sequence_mixer(cfg: "ModelConfig", layer_idx: int) -> nn.Module:
    """Build the sequence mixer selected by `cfg.mixer`."""
    if cfg.mixer == "mamba2":
        return Mamba2(cfg.d_model, d_state=cfg.d_state, d_conv=cfg.d_conv, expand=cfg.expand,
                      headdim=cfg.headdim, ngroups=cfg.ngroups, layer_idx=layer_idx)
    if cfg.mixer == "gdn2":
        return GatedDeltaNet2Mixer(cfg.d_model, head_dim=cfg.gdn2_head_dim,
                                   num_heads=cfg.gdn2_num_heads, expand_v=cfg.gdn2_expand_v,
                                   mode=cfg.gdn2_mode, use_short_conv=cfg.gdn2_use_short_conv,
                                   conv_size=cfg.gdn2_conv_size,
                                   allow_neg_eigval=cfg.gdn2_allow_neg_eigval, layer_idx=layer_idx)
    raise ValueError(f"unknown mixer {cfg.mixer!r} (expected 'mamba2' or 'gdn2')")


class Block(nn.Module):
    """Pre-norm block: norm -> sequence mixer, then norm -> MLP (state mixer)."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.sequence_mixer = _make_sequence_mixer(cfg, layer_idx)
        self.state_mixer = MLP(cfg.d_model, hidden_mult=cfg.mlp_hidden_mult)
        self.dropout1 = nn.Dropout(cfg.embed_dropout if layer_idx == 0 else cfg.resid_dropout)
        self.norm1 = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)
        self.dropout2 = nn.Dropout(cfg.resid_dropout)
        self.norm2 = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)

    def forward(self, hidden_states, residual=None):
        dropped = self.dropout1(hidden_states)
        residual = (dropped + residual) if residual is not None else dropped
        hidden_states = self.norm1(residual)
        hidden_states = self.sequence_mixer(hidden_states)

        dropped = self.dropout2(hidden_states)
        residual = dropped + residual
        hidden_states = self.norm2(residual)
        hidden_states = self.state_mixer(hidden_states)
        return hidden_states, residual


class TokenEmbeddings(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.word_embeddings = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if not cfg.learnable_word_embeddings:
            self.word_embeddings.weight.requires_grad = False
        self.max_position_embeddings = cfg.max_position_embeddings
        if self.max_position_embeddings > 0:
            self.position_embeddings = nn.Embedding(cfg.max_position_embeddings, cfg.d_model)

    def forward(self, input_ids):
        emb = self.word_embeddings(input_ids)
        if self.max_position_embeddings > 0:
            pos = torch.arange(input_ids.shape[1], dtype=torch.long, device=input_ids.device)
            emb = emb + self.position_embeddings(pos)
        return emb


def _init_weights(module, n_layers, initializer_range=0.02):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, std=initializer_range)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        if not getattr(module, "_no_reinit", False):
            nn.init.normal_(module.weight, std=initializer_range)
    # GPT-2-style residual rescaling for projections feeding the residual stream
    for name, p in module.named_parameters():
        if name in ("out_proj.weight", "fc2.weight"):
            nn.init.normal_(p, mean=0.0, std=initializer_range / math.sqrt(2 * n_layers))


class LanguageModel(nn.Module):
    """Mirrors zoology.model.LanguageModel (sequence_mixer selected by cfg.mixer)."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.embeddings = TokenEmbeddings(cfg)
        self.layers = nn.ModuleList([Block(cfg, layer_idx=i) for i in range(cfg.n_layers)])
        self.drop_f = nn.Dropout(cfg.resid_dropout)
        self.ln_f = nn.LayerNorm(cfg.d_model, eps=cfg.layer_norm_epsilon)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        from functools import partial
        init_fn = partial(_init_weights, n_layers=cfg.n_layers, initializer_range=cfg.initializer_range)
        for name, module in self.named_modules():
            if "sequence_mixer.gdn2" in name:   # fla GatedDeltaNet2 self-initializes; don't override it
                continue
            init_fn(module)

        if cfg.learnable_word_embeddings:  # weight tying
            self.lm_head.weight = self.embeddings.word_embeddings.weight

    def forward(self, input_ids):
        hidden_states = self.embeddings(input_ids)
        residual = None
        for layer in self.layers:
            hidden_states, residual = layer(hidden_states, residual)
        dropped = self.drop_f(hidden_states)
        residual = dropped + residual
        hidden_states = self.ln_f(residual)
        return self.lm_head(hidden_states)

    def state_size(self, sequence_length: int) -> int:
        # zoology convention: sum the per-layer recurrent state, x4 bytes (float32)
        total = sum(layer.sequence_mixer.state_size(sequence_length) for layer in self.layers)
        return 4 * total


def mamba2_state_size(d_model: int, n_layers: int, d_state: int, expand: int = 2) -> int:
    """Closed form of LanguageModel.state_size for Mamba2: 4 * n_layers * (expand*d_model) * d_state.

    Independent of sequence length (bounded recurrent state)."""
    return 4 * n_layers * (expand * d_model) * d_state


def gdn2_state_size(d_model: int, n_layers: int, head_dim: int, num_heads: int = None,
                    expand_v: float = 1.0) -> int:
    """Closed form of LanguageModel.state_size for Gated DeltaNet 2:
    4 * n_layers * num_heads * head_dim * head_v_dim  (head_v_dim = head_dim * expand_v).

    Independent of sequence length (bounded recurrent state)."""
    if num_heads is None:
        num_heads = max(1, d_model // head_dim)
    head_v_dim = int(head_dim * expand_v)
    return 4 * n_layers * num_heads * head_dim * head_v_dim


def d_model_for_state_size(target_state: float, n_layers: int, d_state: int, expand: int = 2) -> int:
    """Invert the Mamba2 state-size formula to pick the d_model nearest `target_state` (fixed d_state)."""
    raw = target_state / (4 * n_layers * expand * d_state)
    return max(1, int(round(raw)))

## 4. Training loop
Based on `zoology/train.py`: AdamW, cross-entropy with `ignore_index=-100` (so only answer
positions count), per-example recall accuracy over non-ignored positions. As before:
* **Per-step LR schedule:** linear **warmup over 1 epoch**, then cosine decay to 0. The
  peak LR is the width-scaled `lr_for_d_model`.
* **Gradient clipping** at `max_norm=1.0`.
* **Early stopping** when the task is solved (`valid/accuracy > 0.99`) or when validation
  accuracy plateaus for `patience` epochs.
* **Mixed precision:** for `gdn2` on GPU the forward runs under `torch.autocast(bfloat16)`
  (the fla Triton kernels expect bf16/fp16 inputs); parameters stay fp32 and the loss is
  computed in fp32. `mamba2` runs in full fp32 (its pure-PyTorch scan is precision-sensitive).

In [ ]:
def compute_metrics(preds, targets, ignore_index=-100):
    """Per-example recall accuracy over non-ignored positions (zoology.train.compute_metrics)."""
    accs = []
    for pred, target in zip(preds, targets):
        mask = target != ignore_index
        accs.append((pred[mask] == target[mask]).float().mean().item())
    return accs


def train_one_run(model: LanguageModel, train_dl, test_dl, logger, peak_lr: float):
    model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()  # default ignore_index = -100

    # gdn2's fla Triton kernels expect bf16/fp16 inputs: run the forward under autocast
    # (params stay fp32, loss in fp32). mamba2's pure-PyTorch scan stays full fp32.
    use_amp = (MIXER == "gdn2" and DEVICE == "cuda")
    amp_dtype = torch.bfloat16

    def forward_logits(inputs):
        with torch.autocast(device_type=DEVICE, dtype=amp_dtype, enabled=use_amp):
            return model(inputs)

    # No weight decay on 1-D params (A_log, dt_bias, D, norm/bias) -- standard practice.
    decay = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": TRAIN.weight_decay},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=peak_lr,
    )

    # Per-step schedule: linear warmup over `warmup_epochs`, then cosine decay to 0.
    steps_per_epoch = len(train_dl)
    warmup_steps = max(1, int(TRAIN.warmup_epochs * steps_per_epoch))
    total_steps = max(warmup_steps + 1, TRAIN.max_epochs * steps_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:                       # linear 0 -> 1
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)  # cosine 1 -> 0
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_acc, epochs_no_improve = 0.0, 0
    final_metrics = {}
    for epoch in range(TRAIN.max_epochs):
        # ---- train ----
        model.train()
        for inputs, targets in tqdm(train_dl, desc=f"train {epoch+1}/{TRAIN.max_epochs}", leave=False):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            logits = forward_logits(inputs)
            loss = loss_fn(rearrange(logits, "... c -> (...) c").float(), targets.flatten())
            loss.backward()
            if TRAIN.grad_clip and TRAIN.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN.grad_clip)
            optimizer.step()
            scheduler.step()
            logger.log({"train/loss": loss.item(), "lr": scheduler.get_last_lr()[0], "epoch": epoch})

        # ---- eval ----
        model.eval()
        test_loss, accs = 0.0, []
        with torch.no_grad():
            for inputs, targets in test_dl:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                logits = forward_logits(inputs)
                loss = loss_fn(rearrange(logits, "... c -> (...) c").float(), targets.flatten())
                test_loss += loss.item() / len(test_dl)
                accs.extend(compute_metrics(logits.argmax(-1).cpu(), targets.cpu()))

        acc = float(np.mean(accs))
        if acc > best_acc:
            best_acc, epochs_no_improve = acc, 0
        else:
            epochs_no_improve += 1
        final_metrics = {"valid/loss": test_loss, "valid/accuracy": acc, "valid/best_accuracy": best_acc}
        logger.log({"epoch": epoch, **final_metrics})
        print(f"  epoch {epoch+1:>3}/{TRAIN.max_epochs}  valid/loss={test_loss:.4f}  "
              f"valid/accuracy={acc:.4f}  best={best_acc:.4f}  no_improve={epochs_no_improve}")

        # ---- early stopping: solved, or no val-accuracy improvement for `patience` epochs ----
        if acc > TRAIN.early_stopping_threshold:
            print(f"  early stop (solved): valid/accuracy {acc:.4f} > {TRAIN.early_stopping_threshold}")
            break
        if epochs_no_improve >= TRAIN.patience:
            print(f"  early stop (plateau): no valid/accuracy improvement for {TRAIN.patience} "
                  f"epochs (best={best_acc:.4f})")
            break

    final_metrics["valid/best_accuracy"] = best_acc
    return final_metrics

## 5. Weights & Biases setup
We log in (skipped if `WANDB_MODE` is `disabled`/`offline`). The helper below assembles
the **complete nested config** — task, model, training, derived state size, seed, and
the runtime/library environment — which is what gets stored in W&B.

In [ ]:
import wandb

if WANDB_MODE not in ("disabled", "offline"):
    try:
        wandb.login()
    except Exception as e:
        print(f"wandb.login() failed ({e}); falling back to offline mode.")
        WANDB_MODE = "offline"


def build_full_config(model_cfg: ModelConfig, state_size: int,
                      num_parameters: int, fingerprint: str, sweep_id: str, peak_lr: float) -> dict:
    """The full, reproducible config saved to W&B."""
    return {
        "architecture": MIXER,
        "task": asdict(TASK),
        "model": asdict(model_cfg),
        "train": asdict(TRAIN),
        "sweep": {
            "sweep_id": sweep_id,
            "mixer": MIXER,
            "d_models": list(D_MODELS),
            "n_points": N_POINTS,
            "lr_rule": {
                "type": "clamped_inverse_width",
                "lr_ref": LR_REF, "d_ref": D_REF, "lr_min": LR_MIN, "lr_max": LR_MAX,
            },
        },
        "derived": {
            "state_size": state_size,
            "state_size_seq_len": TASK.input_seq_len,
            "num_parameters": num_parameters,
            "peak_learning_rate": peak_lr,
        },
        "reproducibility": {
            "seed": SEED,
            "dataset_fingerprint": fingerprint,
            "torch_version": torch.__version__,
            "numpy_version": np.__version__,
            "python_version": platform.python_version(),
            "device": DEVICE,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cudnn_deterministic": True,
        },
    }


class WandbLogger:
    """Thin logger mirroring zoology.logger.WandbLogger (no-ops if W&B is disabled)."""
    def __init__(self, run):
        self.run = run

    def log(self, metrics: dict):
        if self.run is not None:
            self.run.log(metrics)

## 6. Run the sweep
We train one model per width in `D_MODELS` with the selected `MIXER`. Each width maps to a
state size (the x-axis) and uses its own width-scaled peak learning rate (`lr_for_d_model`,
clamped 1/width). Every run re-seeds from the same global `SEED` (so the dataset and init
are deterministic) and saves its full config to W&B.

In [ ]:
SWEEP_ID = uuid.uuid4().hex[:8]
GROUP = f"mqar-{MIXER}-exp003"

# Model knobs fixed by the (base) config; each width -> a state size (the x-axis).
N_LAYERS = ModelConfig().n_layers
d_models = list(D_MODELS)


def planned_point(d_model: int):
    """(state_size, dims_str) for the planning table, per mixer (no model is built)."""
    cfg = ModelConfig(d_model=d_model)
    if MIXER == "mamba2":
        d_inner = cfg.expand * d_model
        hd = _valid_headdim(d_inner, cfg.headdim)
        state = mamba2_state_size(d_model, N_LAYERS, cfg.d_state, cfg.expand)
        dims = f"d_inner={d_inner:>4d}  headdim={hd:>3d}  nheads={d_inner // hd:>3d}"
    else:  # gdn2
        nh, hd = cfg.gdn2_num_heads, cfg.gdn2_head_dim
        hv = int(hd * cfg.gdn2_expand_v)
        state = gdn2_state_size(d_model, N_LAYERS, hd, nh, cfg.gdn2_expand_v)
        dims = f"num_heads={nh:>3d}  head_dim={hd:>3d}  head_v_dim={hv:>3d}"
    return state, dims


state_sizes = [planned_point(d)[0] for d in d_models]

print(f"Planned sweep (mixer={MIXER!r}; state size is the x-axis):")
for d in d_models:
    state, dims = planned_point(d)
    print(f"  d_model={d:>4d}  {dims}  ->  state_size={state:>11d}  ->  lr={lr_for_d_model(d):.2e}")

results = []
for i, d_model in enumerate(d_models):
    peak_lr = lr_for_d_model(d_model)
    print(f"\n=== Run {i+1}/{N_POINTS}: d_model={d_model} (lr={peak_lr:.2e}) ===")
    set_determinism(SEED)

    # data (identical task across runs) + model
    train_dl, test_dl, fingerprint = build_dataloaders(TASK, seed=SEED)
    model_cfg = ModelConfig(d_model=d_model, vocab_size=TASK.vocab_size,
                            max_position_embeddings=TASK.input_seq_len)
    model = LanguageModel(model_cfg)

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    state_size = model.state_size(sequence_length=TASK.input_seq_len)
    full_config = build_full_config(model_cfg, state_size, num_params,
                                    fingerprint, SWEEP_ID, peak_lr)

    run = None
    if WANDB_MODE != "disabled":
        run = wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY,
            name=f"d_model{d_model}-state{state_size}", group=GROUP, job_type="train",
            mode=WANDB_MODE, config=full_config, reinit=True,
            tags=["mqar", MIXER, "state-size-sweep"],
        )
        run.log({"state_size": state_size, "num_parameters": num_params})
        # also persist the full config as a downloadable artifact for full reproducibility
        with open("config.json", "w") as f:
            json.dump(full_config, f, indent=2)
        run.save("config.json")

    logger = WandbLogger(run)
    metrics = train_one_run(model, train_dl, test_dl, logger, peak_lr=peak_lr)

    if run is not None:
        run.summary.update({"state_size": state_size, "num_parameters": num_params,
                            "peak_learning_rate": peak_lr, **metrics})
        run.finish()

    results.append({
        "d_model": d_model,
        "state_size": int(state_size),
        "num_parameters": int(num_params),
        "learning_rate": peak_lr,
        "valid_accuracy": metrics["valid/accuracy"],
        "valid_best_accuracy": metrics["valid/best_accuracy"],
    })

print("\nSweep complete.")
for r in results:
    print(f"  state_size={r['state_size']:>11d}  d_model={r['d_model']:>4d}  "
          f"lr={r['learning_rate']:.2e}  best_acc={r['valid_best_accuracy']:.4f}")

## 7. The recreated plot: state size vs. recall accuracy
Accuracy on a linear y-axis against state size on a **log** x-axis — the canonical
zoology view. The figure is saved to disk, displayed inline, and logged to a dedicated
W&B **summary** run as a matplotlib image, a `wandb.Table`, and a native line plot.

In [ ]:
import matplotlib.pyplot as plt

MIXER_LABEL = {"mamba2": "Mamba2", "gdn2": "Gated DeltaNet 2"}[MIXER]
MIXER_COLOR = {"mamba2": "#c4694b", "gdn2": "#4b78c4"}[MIXER]

results_sorted = sorted(results, key=lambda r: r["state_size"])
xs = [r["state_size"] for r in results_sorted]
ys = [r["valid_best_accuracy"] for r in results_sorted]   # best val accuracy (robust to overfitting tail)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xs, ys, marker="o", linewidth=2, markersize=8, color=MIXER_COLOR, label=MIXER_LABEL)
ax.set_xscale("log")
ax.set_xlabel("State size (bytes)")
ax.set_ylabel("MQAR recall accuracy (best)")
ax.set_ylim(-0.02, 1.02)
ax.set_title(f"MQAR: State Size vs. Recall Accuracy\n({MIXER_LABEL}, seq_len={TASK.input_seq_len}, "
             f"{TASK.num_kv_pairs} KV pairs, vocab={TASK.vocab_size})")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
for x, y, d in zip(xs, ys, [r["d_model"] for r in results_sorted]):
    ax.annotate(f"d={d}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
fig.tight_layout()
fig.savefig("mqar_state_size_vs_accuracy.png", dpi=150)
plt.show()

if WANDB_MODE != "disabled":
    summary = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                         name=f"summary-{SWEEP_ID}", group=GROUP, job_type="summary",
                         mode=WANDB_MODE, reinit=True, tags=["mqar", MIXER, "summary"])
    table = wandb.Table(
        columns=["state_size", "valid_best_accuracy", "valid_accuracy", "d_model",
                 "learning_rate", "num_parameters"],
        data=[[r["state_size"], r["valid_best_accuracy"], r["valid_accuracy"], r["d_model"],
               r["learning_rate"], r["num_parameters"]] for r in results_sorted])
    summary.log({
        "state_size_vs_accuracy_plot": wandb.Image(fig),
        "results": table,
        "state_size_vs_accuracy": wandb.plot.line(table, "state_size", "valid_best_accuracy",
                                                  title="MQAR: state size vs recall accuracy"),
    })
    summary.finish()
    print("Logged summary plot + table to Weights & Biases.")

## 8. Notes & extensions
* **Expected shape.** Both mixers have a **bounded** recurrent state, so recall accuracy is
  capped by state size: a small state cannot hold all `num_kv_pairs` bindings over the large
  vocabulary, and accuracy rises with state size toward 1.0 — the headline zoology
  *recall-vs-state* curve for sub-quadratic mixers.
* **Comparing mixers.** Run the sweep once with `MIXER="mamba2"` and once with `"gdn2"`
  (same task, harness and seed) and overlay the two curves. Gated DeltaNet's delta-rule
  update is generally more recall-efficient per state byte than a diagonal SSM.
* **State knobs.** Mamba2: sweep `d_model` (or `d_state`) — both enter `expand·d_model·d_state`
  linearly. GDN2: state is `num_heads·head_dim·head_v_dim`; here `num_heads = d_model//head_dim`
  scales with width. You can instead grow `gdn2_head_dim` or `gdn2_expand_v` to isolate
  per-head memory.
* **Learning rate.** We reuse the width-scaled peak LR (`lr_for_d_model`, clamped 1/width).
  The paper instead grid-searches the LR per point; for the most faithful result, wrap the
  sweep loop in `for lr in ...` and keep the best.
* **Kernels / precision.** `gdn2` uses fla's Triton `chunk`/`fused_recurrent` kernels (GPU
  only) and runs the forward in bf16. `mamba2` is exact O(L²) pure-PyTorch fp32 (CPU or GPU).
* **Harder task.** Increase `num_kv_pairs` / `input_seq_len` in `TASK` to push the transition
  region; with bounded state the degradation is more pronounced than for attention.